In [ ]:
# How many polymarket markets exist?
#
# The Gamma API is what powers polymarket.com's frontend, so it's the
# right universe for "markets a user would actually see."
#
# Gotchas learned the hard way:
#   - `limit` is capped at 100 (asking for 500 silently returns 100).
#   - `offset` is capped at ~2000 (offset >= 2100 returns HTTP 422).
# To get past the offset cap we paginate several filter "buckets"
# (closed=false, closed=true, archived=true) and union by market id.
import requests
import pandas as pd
import time

BASE = "https://gamma-api.polymarket.com/markets"
PAGE = 100

def fetch_bucket(**filters):
    rows, offset = [], 0
    while True:
        params = {"limit": PAGE, "offset": offset, **filters}
        r = requests.get(BASE, params=params, timeout=60)
        if r.status_code == 422:
            print(f"  hit offset cap at {offset} for {filters}")
            break
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        if len(batch) < PAGE:
            break
        time.sleep(0.02)
    return rows

buckets = [
    dict(closed="false", archived="false"),
    dict(closed="true",  archived="false"),
    dict(archived="true"),
]
by_id = {}
for f in buckets:
    got = fetch_bucket(**f)
    print(f"filter={f}: {len(got):,}")
    for m in got:
        by_id[m["id"]] = m

all_markets = list(by_id.values())
print(f"\nTotal UNIQUE Gamma markets: {len(all_markets):,}")

In [ ]:
# Build a DataFrame and break the count down by status
df = pd.DataFrame(all_markets)
print(f"Total: {len(df):,}")
print()
print("By status:")
print(f"  active=True:   {df['active'].sum():,}")
print(f"  closed=True:   {df['closed'].sum():,}")
print(f"  archived=True: {df['archived'].sum():,}")
print(f"  active & !closed & !archived (live): "
      f"{((df['active']) & (~df['closed']) & (~df['archived'])).sum():,}")
df.head()

In [ ]:
# List of all markets - keep the useful columns and save to CSV
cols = ["id", "slug", "question", "active", "closed", "archived",
        "startDate", "endDate", "volumeNum", "liquidityNum"]
markets_list = df[cols].copy()
markets_list.to_csv("data/polymarket_gamma_markets.csv", index=False)
print(f"Saved {len(markets_list):,} markets to data/polymarket_gamma_markets.csv")
markets_list.head(20)